# 12 - Train Symile bridge ablations

All variants use identical HuBERT-ECG and RAD-DINO features, patient-disjoint Symile splits, optimizer settings, and seeds. The shuffled-pair run is a negative control. By default all configured variants and three seeds run; environment variables can restrict a smoke test.

Shared-specific variants reconstruct normalized foundation features from their quantized shared and specific components. This prevents the specific branch from becoming an unused orthogonal residual. Checkpoints are selected by mean bidirectional Recall@1 on the ordinary Symile validation split; the test split is evaluated only after training.

This notebook uses only the Symile manifests and foundation features produced by notebooks 02-04. It does not consume the external ECG or CXR manifests from notebooks 10-11; those are first used by notebook 13.

- `ABLATION_VARIANTS=clip_continuous,shared_vq`
- `ABLATION_SEEDS=43`
- `ABLATION_EPOCHS=2`


In [ ]:
from pathlib import Path
import importlib
import json
import os
import random
import sys
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

ARTIFACT_ROOT = Path(os.environ.get("ECG_CXR_BRIDGE_ARTIFACT_ROOT", "/path/to/ecg_cxr_bridge_artifacts"))
FEATURE_ROOT, CODE_ROOT = ARTIFACT_ROOT / "features", ARTIFACT_ROOT / "code"
CODE_ROOT.mkdir(parents=True, exist_ok=True)
config = json.loads((ARTIFACT_ROOT / "config.json").read_text())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(DEVICE), "artifact_root": str(ARTIFACT_ROOT)})


In [ ]:
module_source = r'''
from dataclasses import dataclass
import torch
from torch import nn
from torch.nn import functional as F

MODALITIES = ("ecg", "cxr")

class CompositionalCodebook(nn.Module):
    def __init__(self, size, code_dim, temperature=0.1):
        super().__init__()
        self.code_dim, self.temperature = int(code_dim), float(temperature)
        self.codes = nn.Parameter(torch.randn(size, code_dim))
        nn.init.normal_(self.codes, std=code_dim ** -0.5)
    def forward(self, features):
        chunks = features.reshape(features.shape[0], -1, self.code_dim)
        logits = torch.einsum("bnd,kd->bnk", F.normalize(chunks, dim=-1), F.normalize(self.codes, dim=-1))
        indices = logits.argmax(-1)
        hard = F.embedding(indices, self.codes)
        commitment = F.mse_loss(chunks, hard.detach()) + F.mse_loss(hard, chunks.detach())
        quantized = chunks + (hard - chunks).detach()
        # Retain the compositional-chunk axis [batch, chunks, codes].
        # Averaging chunks before code matching destroys positional code information.
        probabilities = F.softmax(logits / self.temperature, dim=-1)
        return quantized.flatten(1), indices, probabilities, commitment

@dataclass
class Output:
    common: torch.Tensor
    continuous_common: torch.Tensor
    common_indices: object = None
    common_probabilities: object = None
    specific: object = None
    continuous_specific: object = None
    specific_indices: object = None
    commitment: object = None
    reconstruction: object = None

class AblationBridge(nn.Module):
    def __init__(self, input_dims, variant, common_dim=256, specific_dim=256, code_dim=8,
                 common_codebook_size=512, specific_codebook_size=256,
                 code_temperature=0.1, dropout=0.1):
        super().__init__()
        allowed = {"clip_continuous", "shared_vq", "shared_specific_vq",
                   "shared_specific_no_code_match", "shuffled_pairs_control"}
        if variant not in allowed:
            raise ValueError(f"Unknown variant {variant!r}; expected one of {sorted(allowed)}")
        if common_dim % code_dim or (variant in {"shared_specific_vq", "shared_specific_no_code_match", "shuffled_pairs_control"} and specific_dim % code_dim):
            raise ValueError("common_dim and specific_dim must be divisible by code_dim")
        self.variant, self.common_dim = variant, int(common_dim)
        self.uses_vq = variant != "clip_continuous"
        self.uses_specific = variant in {"shared_specific_vq", "shared_specific_no_code_match", "shuffled_pairs_control"}
        output_dim = common_dim + (specific_dim if self.uses_specific else 0)
        self.projectors = nn.ModuleDict({name: nn.Sequential(
            nn.LayerNorm(input_dims[name]), nn.Dropout(dropout), nn.Linear(input_dims[name], output_dim)
        ) for name in MODALITIES})
        self.common_codebook = CompositionalCodebook(common_codebook_size, code_dim, code_temperature) if self.uses_vq else None
        self.specific_codebooks = nn.ModuleDict({name: CompositionalCodebook(
            specific_codebook_size, code_dim, code_temperature) for name in MODALITIES}) if self.uses_specific else None
        self.decoders = nn.ModuleDict({name: nn.Sequential(
            nn.LayerNorm(common_dim + specific_dim), nn.Linear(common_dim + specific_dim, input_dims[name])
        ) for name in MODALITIES}) if self.uses_specific else None

    def encode_modality(self, name, features):
        projected = self.projectors[name](features)
        continuous_common = projected[:, :self.common_dim]
        commitments = []
        if self.uses_vq:
            common, common_indices, probabilities, commitment = self.common_codebook(continuous_common)
            commitments.append(commitment)
        else:
            common, common_indices, probabilities = continuous_common, None, None
            commitment = continuous_common.sum() * 0.0
        specific = continuous_specific = specific_indices = reconstruction = None
        if self.uses_specific:
            continuous_specific = projected[:, self.common_dim:]
            specific_raw, specific_indices, _, specific_commitment = self.specific_codebooks[name](continuous_specific)
            commitments.append(specific_commitment)
            reconstruction = self.decoders[name](torch.cat([common, specific_raw], dim=-1))
            specific = F.normalize(specific_raw, dim=-1)
        if commitments:
            # Average within modality so full CodeBind is not penalized twice
            # merely because it has both shared and specific quantizers.
            commitment = torch.stack(commitments).mean()
        return Output(F.normalize(common, dim=-1), continuous_common, common_indices,
                      probabilities, specific, continuous_specific, specific_indices,
                      commitment, reconstruction)

    def forward(self, batch):
        return {name: self.encode_modality(name, batch[name]) for name in MODALITIES}

def symmetric_info_nce(left, right, subject_id, temperature):
    logits = left @ right.T / temperature
    same = subject_id[:, None].eq(subject_id[None, :])
    offdiag = ~torch.eye(len(subject_id), dtype=torch.bool, device=left.device)
    logits = logits.masked_fill(same & offdiag, -torch.inf)
    target = torch.arange(len(left), device=left.device)
    return .5 * (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target))

def js(left, right):
    eps = torch.finfo(left.dtype).eps
    middle = .5 * (left + right)
    return .5 * ((left * ((left + eps).log() - (middle + eps).log())).sum(-1)
                 + (right * ((right + eps).log() - (middle + eps).log())).sum(-1)).mean()

def orthogonality(shared, specific):
    if len(shared) < 2:
        return shared.sum() * 0.0
    shared = (shared - shared.mean(0)) / shared.std(0, unbiased=False).clamp_min(1e-4)
    specific = (specific - specific.mean(0)) / specific.std(0, unbiased=False).clamp_min(1e-4)
    cross_correlation = shared.T @ specific / len(shared)
    return cross_correlation.square().mean()

def loss(model, outputs, batch, weights):
    zero = outputs["ecg"].common.sum() * 0.0
    alignment = symmetric_info_nce(outputs["ecg"].common, outputs["cxr"].common,
                                    batch["subject_id"], weights["temperature"])
    relation = js(outputs["ecg"].common_probabilities, outputs["cxr"].common_probabilities) if model.uses_vq else zero
    commitment = torch.stack([outputs[m].commitment for m in MODALITIES]).mean() if model.uses_vq else zero
    if model.uses_specific:
        orth = torch.stack([orthogonality(outputs[m].continuous_common,
                                          outputs[m].continuous_specific) for m in MODALITIES]).mean()
        reconstruction = torch.stack([F.mse_loss(
            outputs[m].reconstruction,
            F.layer_norm(batch[m], (batch[m].shape[-1],))) for m in MODALITIES]).mean()
    else:
        orth = zero
        reconstruction = zero
    if model.uses_vq:
        probs = torch.stack([outputs[m].common_probabilities.mean(0) for m in MODALITIES])
        usage = (probs * (probs + torch.finfo(probs.dtype).eps).log()).sum(-1).mean()
    else:
        usage = zero
    relation_weight = 0.0 if model.variant == "shared_specific_no_code_match" else weights["code_relation_weight"]
    total = (alignment + relation_weight * relation
             + weights["commitment_weight"] * commitment
             + weights["orthogonality_weight"] * orth
             + weights["reconstruction_weight"] * reconstruction
             + weights["code_usage_weight"] * usage)
    return total, {"total": total, "alignment": alignment, "code_relation": relation,
                   "commitment": commitment, "orthogonality": orth,
                   "reconstruction": reconstruction, "code_usage": usage}
'''
module_path = CODE_ROOT / "ecg_cxr_ablation.py"
module_path.write_text(module_source)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_ablation
importlib.reload(ecg_cxr_ablation)
print("Wrote and loaded:", module_path)


In [ ]:
class PairDataset(Dataset):
    def __init__(self, split, shuffle_cxr=False, seed=43):
        paths = {"ecg": FEATURE_ROOT / split / "ecg_hubert.npy",
                 "cxr": FEATURE_ROOT / split / "cxr_raddino.npy",
                 "manifest": ARTIFACT_ROOT / "manifests" / f"{split}.jsonl"}
        missing = [str(path) for path in paths.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError("Run notebooks 02-04 first; missing:\n" + "\n".join(missing))
        self.ecg = np.load(paths["ecg"], mmap_mode="r")
        self.cxr = np.load(paths["cxr"], mmap_mode="r")
        manifest = [json.loads(line) for line in paths["manifest"].read_text().splitlines() if line.strip()]
        if not manifest:
            raise ValueError(f"Empty manifest: {paths['manifest']}")
        if self.ecg.ndim != 2 or self.cxr.ndim != 2:
            raise ValueError(f"Expected 2-D feature arrays, got ECG {self.ecg.shape}, CXR {self.cxr.shape}")
        if len(self.ecg) != len(manifest) or len(self.cxr) != len(manifest):
            raise ValueError(f"{split} row mismatch: ECG={len(self.ecg)}, CXR={len(self.cxr)}, manifest={len(manifest)}")
        self.manifest = manifest
        self.subject = np.asarray([row["subject_id"] for row in manifest], dtype="int64")
        self.cxr_index = np.arange(len(manifest))
        if shuffle_cxr:
            rng = np.random.default_rng(seed)
            largest_patient_count = int(np.unique(self.subject, return_counts=True)[1].max())
            if largest_patient_count > len(self.subject) // 2:
                raise RuntimeError("A cross-patient derangement is impossible because one patient owns more than half the rows")
            for _ in range(1000):
                candidate = rng.permutation(len(manifest))
                if np.all(self.subject != self.subject[candidate]):
                    self.cxr_index = candidate
                    break
            else:
                raise RuntimeError("Could not construct a deranged cross-patient CXR permutation")
    def __len__(self): return len(self.subject)
    def __getitem__(self, index):
        return {"ecg": torch.tensor(np.array(self.ecg[index], dtype="float32")),
                "cxr": torch.tensor(np.array(self.cxr[self.cxr_index[index]], dtype="float32")),
                "subject_id": torch.tensor(self.subject[index], dtype=torch.long)}

DEFAULT_VARIANTS = ["clip_continuous", "shared_vq", "shared_specific_vq",
                    "shared_specific_no_code_match", "shuffled_pairs_control"]
VARIANTS = [value.strip() for value in os.environ.get(
    "ABLATION_VARIANTS", ",".join(config.get("ablation_variants", DEFAULT_VARIANTS))).split(",") if value.strip()]
unknown_variants = sorted(set(VARIANTS) - set(DEFAULT_VARIANTS))
if unknown_variants:
    raise ValueError(f"Unknown ABLATION_VARIANTS: {unknown_variants}")
default_seeds = config.get("ablation_seeds", [43, 44, 45])
SEEDS = [int(value) for value in os.environ.get("ABLATION_SEEDS", ",".join(map(str, default_seeds))).split(",")]
EPOCHS = int(os.environ.get("ABLATION_EPOCHS", "50"))
BATCH_SIZE = int(os.environ.get("ABLATION_BATCH_SIZE", "256"))
NUM_WORKERS = int(os.environ.get("ABLATION_NUM_WORKERS", "4"))
PATIENCE = int(os.environ.get("ABLATION_PATIENCE", "8"))
LEARNING_RATE = float(os.environ.get("ABLATION_LR", "3e-4"))
WEIGHT_DECAY = float(os.environ.get("ABLATION_WEIGHT_DECAY", "1e-4"))
if not VARIANTS:
    raise ValueError("ABLATION_VARIANTS must contain at least one variant")
if not SEEDS:
    raise ValueError("ABLATION_SEEDS must contain at least one integer seed")
if EPOCHS < 1 or PATIENCE < 1:
    raise ValueError("ABLATION_EPOCHS and ABLATION_PATIENCE must both be at least 1")
if NUM_WORKERS < 0:
    raise ValueError("ABLATION_NUM_WORKERS cannot be negative")
if BATCH_SIZE < 2:
    raise ValueError("ABLATION_BATCH_SIZE must be at least 2 for InfoNCE")
HPARAMS = {"common_dim": 256, "specific_dim": 256, "code_dim": 8,
           "common_codebook_size": 512, "specific_codebook_size": 256,
           "code_temperature": .1, "dropout": .1}
WEIGHTS = {"temperature": .07, "code_relation_weight": .1, "commitment_weight": .25,
           "orthogonality_weight": .05, "reconstruction_weight": .10,
           "code_usage_weight": .01}
# Validate every input before expensive multi-variant training begins.
preflight = {split: PairDataset(split) for split in ["train", "val", "val_retrieval", "test"]}
preflight_train, preflight_val = preflight["train"], preflight["val"]
subject_overlap = set(preflight_train.subject.tolist()) & set(preflight_val.subject.tolist())
if subject_overlap:
    raise AssertionError(f"Patient leakage between Symile train and val: {list(subject_overlap)[:20]}")

retrieval_groups = {}
for split in ["val_retrieval", "test"]:
    groups = {}
    for row_index, row in enumerate(preflight[split].manifest):
        missing_fields = [key for key in ["label_hadm_id", "label"] if key not in row]
        if missing_fields:
            raise KeyError(f"{split} manifest row {row_index} is missing {missing_fields}")
        groups.setdefault(int(row["label_hadm_id"]), []).append(int(row["label"]))
    invalid = {group_id: {"rows": len(labels), "positives": sum(label == 1 for label in labels)}
               for group_id, labels in groups.items()
               if len(labels) != 10 or sum(label == 1 for label in labels) != 1}
    if invalid:
        raise ValueError(f"Invalid 10-candidate groups in {split}: {dict(list(invalid.items())[:20])}")
    retrieval_groups[split] = len(groups)

for name, dataset in preflight.items():
    sample_index = np.linspace(0, len(dataset) - 1, min(1024, len(dataset)), dtype=int)
    if not np.isfinite(np.asarray(dataset.ecg[sample_index])).all() or not np.isfinite(np.asarray(dataset.cxr[sample_index])).all():
        raise ValueError(f"Non-finite foundation features detected in {name}")
print({"variants": VARIANTS, "seeds": SEEDS, "epochs": EPOCHS,
       "train_rows": len(preflight_train), "val_rows": len(preflight_val),
       "retrieval_groups": retrieval_groups,
       "patient_overlap": len(subject_overlap), "loss_weights": WEIGHTS})


In [ ]:
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    rows = []
    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            if training: optimizer.zero_grad(set_to_none=True)
            total, components = ecg_cxr_ablation.loss(model, model(batch), batch, WEIGHTS)
            if training:
                total.backward(); clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            rows.append({key: float(value.detach().cpu()) for key, value in components.items()})
    if not rows:
        raise RuntimeError("DataLoader produced no batches; reduce ABLATION_BATCH_SIZE")
    return {key: float(np.mean([row[key] for row in rows])) for key in rows[0]}

@torch.inference_mode()
def validation_retrieval(model, loader):
    model.eval(); collected = {"ecg": [], "cxr": [], "subject_id": []}
    for batch in loader:
        encoded = model({m: batch[m].to(DEVICE, non_blocking=True) for m in ["ecg", "cxr"]})
        for modality in ["ecg", "cxr"]:
            collected[modality].append(encoded[modality].common.cpu())
        collected["subject_id"].append(batch["subject_id"])
    values = {key: torch.cat(parts) for key, parts in collected.items()}
    similarity = F.normalize(values["ecg"].float(), dim=-1) @ F.normalize(values["cxr"].float(), dim=-1).T
    same_patient = values["subject_id"][:, None].eq(values["subject_id"][None, :])
    off_diagonal = ~torch.eye(len(similarity), dtype=torch.bool)
    similarity = similarity.masked_fill(same_patient & off_diagonal, -torch.inf)
    diagonal = similarity.diag()
    ecg_ranks = 1 + (similarity > diagonal[:, None]).sum(1)
    cxr_ranks = 1 + (similarity.T > diagonal[:, None]).sum(1)
    return {"ecg_to_cxr_r1": float((ecg_ranks == 1).float().mean()),
            "cxr_to_ecg_r1": float((cxr_ranks == 1).float().mean()),
            "mean_r1": float(.5 * ((ecg_ranks == 1).float().mean() + (cxr_ranks == 1).float().mean()))}

experiment_index = []
for variant in VARIANTS:
    for seed in SEEDS:
        seed_all(seed)
        shuffled = variant == "shuffled_pairs_control"
        train_ds = PairDataset("train", shuffle_cxr=shuffled, seed=seed)
        val_ds = PairDataset("val", shuffle_cxr=False, seed=seed)
        loader_generator = torch.Generator().manual_seed(seed)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  drop_last=len(train_ds) >= BATCH_SIZE,
                                  num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                                  persistent_workers=NUM_WORKERS > 0, generator=loader_generator)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                                persistent_workers=NUM_WORKERS > 0)
        input_dims = {"ecg": int(train_ds.ecg.shape[1]), "cxr": int(train_ds.cxr.shape[1])}
        model_variant = "shared_specific_vq" if shuffled else variant
        model = ecg_cxr_ablation.AblationBridge(input_dims, model_variant, **HPARAMS).to(DEVICE)
        if shuffled: model.variant = "shuffled_pairs_control"
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        run_dir = ARTIFACT_ROOT / "ablation" / variant / f"seed_{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        resolved = {"variant": variant, "model_variant": model_variant, "seed": seed,
                    "input_dims": input_dims, "model_hparams": HPARAMS, "loss_weights": WEIGHTS,
                    "trainable_parameters": int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))}
        resolved.update({"learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                         "batch_size": BATCH_SIZE, "selection_metric": "validation_mean_recall_at_1"})
        best_r1, best_loss, stale, history = -1.0, float("inf"), 0, []
        for epoch in range(EPOCHS):
            started = time.time()
            train_metrics = run_epoch(model, train_loader, optimizer)
            val_metrics = run_epoch(model, val_loader)
            val_retrieval = validation_retrieval(model, val_loader)
            row = {"epoch": epoch, "seconds": time.time() - started,
                   "train": train_metrics, "validation": val_metrics,
                   "validation_retrieval": val_retrieval}
            history.append(row); print(variant, seed, json.dumps(row))
            checkpoint = {"epoch": epoch, "model": model.state_dict(), "resolved_config": resolved}
            torch.save(checkpoint, run_dir / "last.pt")
            improved = (val_retrieval["mean_r1"] > best_r1
                        or (val_retrieval["mean_r1"] == best_r1 and val_metrics["total"] < best_loss))
            if improved:
                best_r1, best_loss, stale = val_retrieval["mean_r1"], val_metrics["total"], 0
                torch.save(checkpoint, run_dir / "best.pt")
            else:
                stale += 1
                if stale >= PATIENCE: break
        (run_dir / "metrics.json").write_text(json.dumps(history, indent=2))
        experiment_index.append({"variant": variant, "seed": seed,
                                 "best_validation_retrieval_r1": best_r1,
                                 "best_validation_loss": best_loss,
                                 "checkpoint": str(run_dir / "best.pt")})
index_path = ARTIFACT_ROOT / "ablation" / "experiment_index.json"
index_path.write_text(json.dumps(experiment_index, indent=2))
print("Saved:", index_path)


In [ ]:
@torch.inference_mode()
def encode_symile(checkpoint_path, split):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    resolved = checkpoint["resolved_config"]
    model = ecg_cxr_ablation.AblationBridge(resolved["input_dims"], resolved["model_variant"], **resolved["model_hparams"]).to(DEVICE).eval()
    model.load_state_dict(checkpoint["model"])
    dataset = PairDataset(split, shuffle_cxr=False)
    loader = DataLoader(dataset, batch_size=1024, shuffle=False)
    output = {"ecg": [], "cxr": []}
    for batch in loader:
        encoded = model({m: batch[m].to(DEVICE) for m in ["ecg", "cxr"]})
        for modality in ["ecg", "cxr"]: output[modality].append(encoded[modality].common.cpu())
    output = {key: torch.cat(value) for key, value in output.items()}
    manifest = [json.loads(line) for line in (ARTIFACT_ROOT / "manifests" / f"{split}.jsonl").read_text().splitlines()]
    output["label_hadm_id"] = torch.tensor([row["label_hadm_id"] for row in manifest])
    output["label"] = torch.tensor([row["label"] for row in manifest])
    return output

def ten_candidate(encoded, query_name, candidate_name):
    ranks = []
    for hadm_id in torch.unique(encoded["label_hadm_id"]):
        group = torch.where(encoded["label_hadm_id"] == hadm_id)[0]
        positives = group[encoded["label"][group] == 1]
        if len(group) != 10 or len(positives) != 1: raise ValueError(f"Invalid group {int(hadm_id)}")
        positive = positives[0]
        query = F.normalize(encoded[query_name][positive].float(), dim=-1)
        scores = F.normalize(encoded[candidate_name][group].float(), dim=-1) @ query
        positive_position = torch.where(group == positive)[0][0]
        ranks.append(1 + int((scores > scores[positive_position]).sum()))
    ranks = np.asarray(ranks)
    return {"queries": len(ranks), "recall_at_1": float(np.mean(ranks <= 1)),
            "recall_at_5": float(np.mean(ranks <= 5)), "recall_at_10": float(np.mean(ranks <= 10)),
            "mrr": float(np.mean(1 / ranks)), "median_rank": float(np.median(ranks)), "chance_r1": .1}

def supported_ten_candidate(encoded, query_name, candidate_name):
    unique_candidate_counts = []
    for hadm_id in torch.unique(encoded["label_hadm_id"]):
        group = torch.where(encoded["label_hadm_id"] == hadm_id)[0]
        unique_candidate_counts.append(int(torch.unique(encoded[candidate_name][group], dim=0).shape[0]))
    if min(unique_candidate_counts) <= 1:
        return {"skipped": True,
                "reason": f"{candidate_name} is repeated within at least one 10-candidate group",
                "minimum_unique_candidates": min(unique_candidate_counts)}
    result = ten_candidate(encoded, query_name, candidate_name)
    result.update({"skipped": False, "minimum_unique_candidates": min(unique_candidate_counts)})
    return result

symile_results = []
for experiment in experiment_index:
    row = {**experiment}
    for split in ["val_retrieval", "test"]:
        encoded = encode_symile(experiment["checkpoint"], split)
        row[split] = {"ecg_to_cxr": supported_ten_candidate(encoded, "ecg", "cxr"),
                      "cxr_to_ecg": supported_ten_candidate(encoded, "cxr", "ecg")}
    symile_results.append(row)
    print(experiment["variant"], experiment["seed"], row["val_retrieval"])
metrics_path = ARTIFACT_ROOT / "reports" / "ablation" / "symile_bridge_ablation_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(symile_results, indent=2))
summary = {}
for variant in sorted({row["variant"] for row in symile_results}):
    subset = [row for row in symile_results if row["variant"] == variant]
    summary[variant] = {"seeds": len(subset)}
    for split in ["val_retrieval", "test"]:
        summary[variant][split] = {}
        for direction in ["ecg_to_cxr", "cxr_to_ecg"]:
            values = [row[split][direction]["recall_at_1"] for row in subset
                      if not row[split][direction].get("skipped", False)]
            if values:
                summary[variant][split][direction] = {
                    "runs": len(values), "recall_at_1_mean": float(np.mean(values)),
                    "recall_at_1_std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
                }
            else:
                summary[variant][split][direction] = {"runs": 0, "skipped": True,
                                                      "reason": "candidate modality is repeated within groups"}
summary_path = ARTIFACT_ROOT / "reports" / "ablation" / "symile_bridge_ablation_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("Saved:", metrics_path, summary_path)
